# Structured Logging

This notebook covers:

1. Why logs should be JSON, not free-text
2. Configure `structlog` for JSON output with a stdlib-compatible processor chain
3. Attach a per-request `request_id` to every line via a middleware + `contextvars`
4. Pick the right log level — `debug` for noise, `info` for facts, `warning` for recoverable problems, `error` for things that need a human
5. Log to **stdout** in containers, never to files

**Scope**: FastAPI + `structlog` + `TestClient`. Notebook 6.1 promised every error response would carry a `request_id` for log correlation — this notebook is where we make that ID real, and where every log line gets it attached automatically.

## 1. Logs Should Be Structured

Consider three lines from a typical free-text log:

```
INFO  2026-06-13 14:02:11 created asset AAPL price=190 by user=alice
INFO  2026-06-13 14:02:13 created asset MSFT price=420 by user=alice
WARN  2026-06-13 14:02:15 ticker AAPL already exists
```

Now imagine you need to answer:

- *Which users created the most assets in the last hour?* — grep, then regex out the `user=...` field, then count.
- *Which tickers are colliding most often?* — grep `"already exists"`, regex out the ticker, count.
- *What was the price distribution on creates?* — grep, regex, parse to float, histogram.

Every question requires a custom regex. Every format change breaks every query.

Now imagine the same lines as JSON:

```
{"ts": "...", "level": "info", "event": "asset_created", "ticker": "AAPL", "price": 190, "user": "alice"}
{"ts": "...", "level": "info", "event": "asset_created", "ticker": "MSFT", "price": 420, "user": "alice"}
{"ts": "...", "level": "warning", "event": "duplicate_ticker", "ticker": "AAPL"}
```

Every question becomes a Loki / Datadog / Splunk **field query**. Format changes mean a new key, not a broken regex. Numbers stay numbers, not strings. Aggregations work.

Three structural goals:

- **Structured**: every line is a JSON object the log backend can index.
- **Stable event names**: `event="asset_created"`, not free-text. Treat them like error codes (notebook 6.1) — the contract your dashboards depend on.
- **Context that travels**: `request_id`, `user_id`, `trace_id` (notebook 6.3) get attached *automatically* to every line emitted during a request — not manually inside each call site.

## 2. `structlog` Basics

`structlog` is a logging library built around the idea that a log call is a **dict**, and processors transform that dict on its way to the output. A minimal config:

- **Processors** (the chain): add timestamp → add level → add context (next section) → render to JSON.
- **`structlog.get_logger()`**: the entry point. Calls like `log.info("asset_created", ticker="AAPL", price=190)` build a dict from the keyword args; the processors decorate and serialize it.

The library also has a stdlib-compatible mode so libraries that use `import logging` get the same structured output. For the capstone we'll wire that in; here we use the standalone setup to keep the focus on the data flow.

In [ ]:
import io
import logging
import sys
import structlog

# Capture log output to a string so we can assert against it. In a real app you'd write to stdout (section 5).
log_buffer = io.StringIO()

def configure_structlog(stream):
    structlog.configure(
        processors=[
            structlog.contextvars.merge_contextvars,                   # request_id, user_id, ...
            structlog.processors.add_log_level,                        # add 'level' key
            structlog.processors.TimeStamper(fmt="iso", utc=True),     # add 'timestamp' key
            structlog.processors.StackInfoRenderer(),                  # include stack on exception()
            structlog.processors.format_exc_info,                      # turn exc_info into a string
            structlog.processors.JSONRenderer(sort_keys=True),         # final shape: one JSON line
        ],
        wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),
        logger_factory=structlog.PrintLoggerFactory(file=stream),      # write to our buffer
        cache_logger_on_first_use=True,
    )

configure_structlog(log_buffer)
log = structlog.get_logger()

# Three log calls with structured fields.
log.info("asset_created", ticker="AAPL", price=190, user="alice")
log.info("asset_created", ticker="MSFT", price=420, user="alice")
log.warning("duplicate_ticker", ticker="AAPL")

print(log_buffer.getvalue().rstrip())

Each call produces **one line of JSON** — directly indexable by any log aggregator. The processor chain is composable: drop in a new processor (a hostname injector, a sampling filter, a redactor for secrets) and every log line gets it for free.

Three patterns to internalize:

- **`event="asset_created"`** is the *first* positional argument. Treat it like an error code — a stable string your dashboards depend on. Don't write `log.info(f"created asset {ticker}")`.
- **Everything else is a `key=value`**. Numbers stay numbers; types are preserved through JSON.
- **`wrapper_class=structlog.make_filtering_bound_logger(logging.INFO)`** sets the level filter. Debug lines below the threshold are *not constructed* — efficient even when you write a lot of `log.debug(...)` calls.

## 3. Per-Request Context: `request_id` via `contextvars`

Every request needs a unique ID. Two reasons:

- **Correlation across log lines.** A single request can produce 10 log lines (auth check, repo query, business logic, response render). All ten need the same ID so you can pull "everything that happened in request `req_abc123`" out of millions of lines.
- **Body ↔ logs correlation.** Notebook 6.1's error envelope already carries `request_id`. The client reports an error with that ID; you grep your logs; you find the full story.

Where the ID **comes from**: either an incoming `X-Request-ID` header (an upstream proxy / load balancer / client may already have one), or generated server-side per request.

How it travels: **`contextvars`**. A Python context variable acts as a per-task global — set it at the start of a request, every code path that runs during that request sees the same value, regardless of how many `Depends`, services, or repo calls deep it goes. `structlog.contextvars.merge_contextvars` (already in our processor chain) reads any context variable currently set and merges it into every log line.

In [ ]:
import uuid
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from starlette.middleware.base import BaseHTTPMiddleware

# Reset the buffer for this section.
log_buffer.seek(0); log_buffer.truncate()

REQUEST_ID_HEADER = "x-request-id"

class RequestIDMiddleware(BaseHTTPMiddleware):
    """Generate (or accept) a request id, bind it to structlog context, echo it on the response."""
    async def dispatch(self, request: Request, call_next):
        rid = request.headers.get(REQUEST_ID_HEADER) or f"req_{uuid.uuid4().hex[:12]}"
        # Bind to the structlog context: every log line emitted inside this request
        # will carry request_id=rid, regardless of who calls log.info().
        structlog.contextvars.bind_contextvars(request_id=rid)
        try:
            response = await call_next(request)
        finally:
            # Clear context so the variable doesn't leak to the next request on the same task.
            structlog.contextvars.clear_contextvars()
        response.headers[REQUEST_ID_HEADER] = rid
        return response

app = FastAPI()
app.add_middleware(RequestIDMiddleware)

# A trivial "service" log call deep inside business logic — no request_id passed in,
# yet it lands in the line. That's the whole point.
def lookup_asset(ticker: str):
    log.info("asset_lookup", ticker=ticker)
    return {"ticker": ticker, "price": 100.0}

@app.get("/assets/{ticker}")
def get_asset(ticker: str):
    log.info("handler_invoked", route="/assets/{ticker}", ticker=ticker)
    return lookup_asset(ticker)

client = TestClient(app)

# Request 1: no incoming header. The middleware generates one and echoes it back.
r1 = client.get("/assets/AAPL")
rid_echoed = r1.headers[REQUEST_ID_HEADER]
print("response 1 X-Request-ID:", rid_echoed)

# Request 2: client supplies their own header. The middleware preserves it.
r2 = client.get("/assets/MSFT", headers={REQUEST_ID_HEADER: "req_from_client_xyz"})
print("response 2 X-Request-ID:", r2.headers[REQUEST_ID_HEADER])

print("\n--- log output ---")
print(log_buffer.getvalue().rstrip())

Two lines per request, both carrying the same `request_id`. The handler **and** the deep "service" call both produce a line — neither had to remember to pass an ID. That's the contextvars magic: bind once at the request boundary, read implicitly from anywhere in the call stack.

Three production properties worth noticing:

- **The client can supply their own ID.** That's deliberate — a CLI tool or another service can tag a request *before* it hits you, and you preserve the tag end-to-end (request 2). Distributed tracing (notebook 6.3) makes this the trace ID instead.
- **The response echoes the header.** The client can correlate failures to logs without parsing the body. (For errors, the ID is *also* in the envelope from 6.1 — belt and suspenders.)
- **`clear_contextvars()` in a `finally` block.** Web servers reuse worker tasks; if you don't clear, the *next* request on the same task inherits the previous request's ID. That's a fun debugging session. Clear deliberately.

## 4. Choosing a Log Level

Five levels, five rules of thumb:

| Level     | Use for                                                                          |
|-----------|----------------------------------------------------------------------------------|
| `debug`   | Verbose internal state — disabled in prod. "got 12 results from query X."        |
| `info`    | Business events worth keeping in the historical record. "asset created."          |
| `warning` | Something unusual happened; service is still working. "retried 3x, succeeded."   |
| `error`   | A handled error a human will want to look at later. "failed to send notification."|
| `critical`| The service is degraded or going down. Reserved.                                  |

Two anti-patterns the levels exist to fix:

- **Everything at `info`** — your alerts can't distinguish noise from real problems. Dashboards burn.
- **`error` on every 4xx** — a 404 from a typo isn't an error. It's a client-side fact. Reserve `error` for things you'd actually wake up an engineer about.

The level threshold should come from settings (notebook 4.3), so you can run `info` in prod and `debug` in dev with no code change. `structlog.make_filtering_bound_logger(level)` is the filter; pass `logging.DEBUG` to enable everything.

In [ ]:
from pydantic_settings import BaseSettings

class ObservabilitySettings(BaseSettings):
    log_level: str = "INFO"   # override with LOG_LEVEL=DEBUG in dev

settings = ObservabilitySettings()  # would read from env / .env in a real app
level_value = getattr(logging, settings.log_level)

# Reset buffer and re-configure with the chosen level.
log_buffer.seek(0); log_buffer.truncate()
structlog.configure(
    processors=[
        structlog.contextvars.merge_contextvars,
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso", utc=True),
        structlog.processors.JSONRenderer(sort_keys=True),
    ],
    wrapper_class=structlog.make_filtering_bound_logger(level_value),
    logger_factory=structlog.PrintLoggerFactory(file=log_buffer),
    cache_logger_on_first_use=False,  # so reconfiguration takes effect
)
log = structlog.get_logger()

log.debug("sql_query", sql="SELECT * FROM assets", rows=12)   # filtered out at INFO
log.info("asset_created", ticker="AAPL")
log.warning("slow_query", sql="SELECT ...", duration_ms=1200)
log.error("notification_failed", channel="email", reason="smtp timeout")

print("with level =", settings.log_level)
print(log_buffer.getvalue().rstrip())

# Now flip to DEBUG, as you'd do via LOG_LEVEL=DEBUG in dev.
log_buffer.seek(0); log_buffer.truncate()
structlog.configure(
    processors=[
        structlog.contextvars.merge_contextvars,
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso", utc=True),
        structlog.processors.JSONRenderer(sort_keys=True),
    ],
    wrapper_class=structlog.make_filtering_bound_logger(logging.DEBUG),
    logger_factory=structlog.PrintLoggerFactory(file=log_buffer),
    cache_logger_on_first_use=False,
)
log = structlog.get_logger()
log.debug("sql_query", sql="SELECT * FROM assets", rows=12)
log.info("asset_created", ticker="AAPL")
print("\nwith level = DEBUG")
print(log_buffer.getvalue().rstrip())

Same code, two configurations, two views of the same events. In dev you see the SQL. In prod the SQL line is never even constructed — the filter cuts it before any of the processor chain runs.

## 5. Stdout for Containers

**Where the logs go: stdout. Always. Period.** Not a file. Not a syslog socket. Not the cloud-vendor SDK. Just stdout.

Why:

- **The container runtime captures stdout.** Docker, containerd, Kubernetes — every orchestrator captures whatever the process writes to stdout / stderr and forwards it to the platform's log aggregator. Your app is *not* responsible for shipping logs.
- **Stateless containers.** No log directory means no disk that fills up, no file rotation to manage, no "why did my container go OOM at 3 AM" because logrotate died.
- **One concern, one place.** Log aggregation is an *infrastructure* problem. Filebeat / Vector / Promtail / FluentBit reads stdout and ships to Loki / Datadog / CloudWatch. The app stays focused on producing structured lines.

In our processor chain, `PrintLoggerFactory(file=log_buffer)` is the test-friendly stand-in for `PrintLoggerFactory(file=sys.stdout)` — *the* production line.

One thing **not** to do: tag the JSON with ANSI color codes for human readability in prod. In dev, swap `JSONRenderer` for `ConsoleRenderer` and you get colorized human output. In prod, the JSON renderer goes to stdout untouched.


In [ ]:
# Dev vs prod renderer toggle. In a real settings module this'd be a single bool.
def configure_for_environment(env: str, stream):
    if env == "prod":
        renderer = structlog.processors.JSONRenderer(sort_keys=True)
    else:
        renderer = structlog.dev.ConsoleRenderer(colors=False)  # colors off for the demo
    structlog.configure(
        processors=[
            structlog.contextvars.merge_contextvars,
            structlog.processors.add_log_level,
            structlog.processors.TimeStamper(fmt="iso", utc=True),
            renderer,
        ],
        wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),
        logger_factory=structlog.PrintLoggerFactory(file=stream),
        cache_logger_on_first_use=False,
    )

# Dev: human-friendly, key=value, no JSON braces.
log_buffer.seek(0); log_buffer.truncate()
configure_for_environment("dev", log_buffer)
log = structlog.get_logger()
structlog.contextvars.bind_contextvars(request_id="req_demo_1")
log.info("asset_created", ticker="AAPL", price=190)
print("DEV format:")
print(log_buffer.getvalue().rstrip())

# Prod: JSON line, log-aggregator-friendly.
log_buffer.seek(0); log_buffer.truncate()
configure_for_environment("prod", log_buffer)
log = structlog.get_logger()
structlog.contextvars.bind_contextvars(request_id="req_demo_2")
log.info("asset_created", ticker="AAPL", price=190)
print("\nPROD format:")
print(log_buffer.getvalue().rstrip())
structlog.contextvars.clear_contextvars()

Same event, two formats. The aggregator (Loki / Datadog) parses the prod line as JSON and indexes every key. The dev format is colorized in a real terminal — `colors=True` is the default; we turned it off so the demo output is readable here.

The bridge to the standard library: in a real app you also configure `logging.basicConfig(stream=sys.stdout, level=settings.log_level)` and ask structlog to wrap stdlib calls (`structlog.stdlib.LoggerFactory` instead of `PrintLoggerFactory`). Libraries you depend on use stdlib `logging`; this guarantees their output goes through the same chain — single output stream, single format. We skip the stdlib wiring here to keep the demo focused; the capstone's `observability.py` does it once at app startup.

## Key Takeaways

- **JSON, not free text.** Every log line is a dict the aggregator can index. Event names are stable strings; fields are typed.
- **`structlog` processor chain** is composable: `merge_contextvars` → `add_log_level` → `TimeStamper` → `JSONRenderer`. Drop processors in or out; every line gets them.
- **Per-request `request_id`** lives in `contextvars`: bind in a middleware, every `log.info(...)` deep in the call stack picks it up, clear in `finally`. The same ID lands in the error envelope (notebook 6.1) and the response header.
- **Honor incoming `X-Request-ID`** so an upstream proxy or test harness can trace a request end-to-end.
- **Log levels mean something.** `info` for business events; `warning` for retried-and-fine; `error` for needs-a-human. Everything-at-info kills your alerts.
- **Level threshold from settings.** Dev: `DEBUG`. Prod: `INFO`. One env var changes it.
- **Stdout, always.** Containers and orchestrators capture it. No log files, no rotation, no application-side shipping.
- **Capstone tie-in**: `observability.py` will host `configure_logging(settings)` and `RequestIDMiddleware`. Every router emits `event="asset_created"` / `event="order_placed"` etc. with `request_id` already attached for free.

## Exercises

**1. Add `user_id` to every log line.** Extend `RequestIDMiddleware` (or write a second middleware that runs after auth) to call `structlog.contextvars.bind_contextvars(user_id=current_user.username)` once the user has been resolved by `get_current_user` (notebook 5.1). Verify a 200 on a protected route emits a log line with `request_id` *and* `user_id` populated. Anonymous routes (e.g., `/health`) should not have `user_id` set — and they shouldn't, since the middleware binds it only when auth ran.

**2. Trace a request across modules.** Build a tiny chain — `router_handler` → `AssetService.create` → `AssetRepo.insert` — where each layer emits one log line. Without passing `request_id` explicitly anywhere, confirm all three lines share the same `request_id`. Now do the same with two concurrent requests via `httpx.AsyncClient`: confirm the two requests' lines have **distinct** IDs and never collide.

**3. Redact a secret.** Add a custom processor that scrubs any value matching a `^Bearer\s` prefix from any field, replacing it with `"<redacted>"`. Verify a `log.info("upstream_call", auth="Bearer abc123")` emits `auth="<redacted>"`. Sketch in a markdown cell why redaction belongs in the processor chain rather than at every call site. (Hint: how many call sites would you have to fix if a third-party library logs the same secret?)